In [ ]:
!pip -q install \
gensim \
pyLDAvis \
openpyxl

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# NG04-1 — Imports and Paths
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from gensim.models import LdaModel
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

warnings.filterwarnings("ignore")

print("Imports completed.")

Imports completed.


In [ ]:
# ============================================================
# NG04-1B — NewsGroupsTopicEval Paths
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval"
)

DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
VECTOR_DIR = BASE_DIR / "vectors"
ANALYSIS_DIR = BASE_DIR / "analysis"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_FILE = BASE_DIR /"config/config.json"

print("BASE_DIR     :", BASE_DIR)
print("DATA_DIR     :", DATA_DIR)
print("MODEL_DIR    :", MODEL_DIR)
print("VECTOR_DIR   :", VECTOR_DIR)
print("ANALYSIS_DIR :", ANALYSIS_DIR)

assert BASE_DIR.exists(), "NewsGroupsTopicEval folder not found."

print("\n✓ Folder structure validated.")

BASE_DIR     : /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval
DATA_DIR     : /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/data
MODEL_DIR    : /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/models
VECTOR_DIR   : /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/vectors
ANALYSIS_DIR : /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/analysis

✓ Folder structure validated.


In [ ]:
# ============================================================
# NG04-2 — Load Configuration
# ============================================================

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

RANDOM_SEED = CONFIG["random_seed"]
N_FOLDS = CONFIG["n_folds"]

TOPIC_MIN = CONFIG["topic_min"]
TOPIC_MAX = CONFIG["topic_max"]

TOPIC_LIST = list(range(TOPIC_MIN, TOPIC_MAX + 1))

LDA_PASSES = CONFIG["lda_passes"]
LDA_ITERATIONS = CONFIG["lda_iterations"]
LDA_CHUNKSIZE = CONFIG["lda_chunksize"]

ALPHA = CONFIG["lda_alpha"]
ETA = CONFIG["lda_eta"]

LDA_UPDATE_EVERY = CONFIG["lda_update_every"]
LDA_DECAY = CONFIG["lda_decay"]
LDA_OFFSET = CONFIG["lda_offset"]

TOP_WORDS = CONFIG["top_words"]

print("=" * 70)
print("NG04 CONFIGURATION")
print("=" * 70)

print(f"Dataset       : {CONFIG['dataset_name']}")
print(f"Folds         : {N_FOLDS}")
print(f"Topic range   : {TOPIC_MIN}–{TOPIC_MAX}")
print(f"Experiments   : {N_FOLDS * len(TOPIC_LIST)}")
print(f"LDA passes    : {LDA_PASSES}")
print(f"LDA iterations: {LDA_ITERATIONS}")
print(f"Chunksize     : {LDA_CHUNKSIZE}")
print(f"Alpha         : {ALPHA}")
print(f"Eta           : {ETA}")
print(f"Top words     : {TOP_WORDS}")

assert ALPHA == "symmetric"
assert ETA == "symmetric"

print("\n✓ NG04 configuration loaded successfully.")

NG04 CONFIGURATION
Dataset       : 20_Newsgroups
Folds         : 5
Topic range   : 2–100
Experiments   : 495
LDA passes    : 20
LDA iterations: 400
Chunksize     : 2000
Alpha         : symmetric
Eta           : symmetric
Top words     : 20

✓ NG04 configuration loaded successfully.


In [ ]:
# ============================================================
# NG04-3 — Inspect NG03 Outputs
# ============================================================

print("=" * 70)
print("MODEL DIRECTORY")
print("=" * 70)

if MODEL_DIR.exists():
    for p in sorted(MODEL_DIR.rglob("*")):
        if p.is_file():
            print(p.relative_to(BASE_DIR))

print("\n" + "=" * 70)
print("VECTOR DIRECTORY")
print("=" * 70)

if VECTOR_DIR.exists():
    for p in sorted(VECTOR_DIR.rglob("*")):
        if p.is_file():
            print(p.relative_to(BASE_DIR))

print("\n" + "=" * 70)
print("ANALYSIS DIRECTORY")
print("=" * 70)

if ANALYSIS_DIR.exists():
    for p in sorted(ANALYSIS_DIR.rglob("*")):
        if p.is_file():
            print(p.relative_to(BASE_DIR))

MODEL DIRECTORY
models/Fold_1/K_10/lda_model
models/Fold_1/K_10/lda_model.expElogbeta.npy
models/Fold_1/K_10/lda_model.id2word
models/Fold_1/K_10/lda_model.state
models/Fold_1/K_100/lda_model
models/Fold_1/K_100/lda_model.expElogbeta.npy
models/Fold_1/K_100/lda_model.id2word
models/Fold_1/K_100/lda_model.state
models/Fold_1/K_11/lda_model
models/Fold_1/K_11/lda_model.expElogbeta.npy
models/Fold_1/K_11/lda_model.id2word
models/Fold_1/K_11/lda_model.state
models/Fold_1/K_12/lda_model
models/Fold_1/K_12/lda_model.expElogbeta.npy
models/Fold_1/K_12/lda_model.id2word
models/Fold_1/K_12/lda_model.state
models/Fold_1/K_13/lda_model
models/Fold_1/K_13/lda_model.expElogbeta.npy
models/Fold_1/K_13/lda_model.id2word
models/Fold_1/K_13/lda_model.state
models/Fold_1/K_14/lda_model
models/Fold_1/K_14/lda_model.expElogbeta.npy
models/Fold_1/K_14/lda_model.id2word
models/Fold_1/K_14/lda_model.state
models/Fold_1/K_15/lda_model
models/Fold_1/K_15/lda_model.expElogbeta.npy
models/Fold_1/K_15/lda_model.i

In [ ]:
# ============================================================
# NG04-4 — Validate Complete LDA Model Set
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

EXPECTED_EXPERIMENTS = N_FOLDS * len(TOPIC_LIST)

print("=" * 70)
print("NG04 MODEL VALIDATION")
print("=" * 70)

print(f"Expected folds       : {N_FOLDS}")
print(f"Expected K values    : {len(TOPIC_LIST)}")
print(f"Expected experiments : {EXPECTED_EXPERIMENTS}")
print()

# ------------------------------------------------------------
# Find all expected model paths
# ------------------------------------------------------------

model_records = []
missing_models = []

for fold in range(1, N_FOLDS + 1):

    for k in TOPIC_LIST:

        model_path = (
            MODEL_DIR
            / f"Fold_{fold}"
            / f"K_{k}"
            / "lda_model"
        )

        exists = model_path.exists()

        model_records.append({
            "Fold": fold,
            "Topics": k,
            "ModelPath": str(model_path),
            "Exists": exists
        })

        if not exists:
            missing_models.append((fold, k))

model_validation_df = pd.DataFrame(model_records)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_found = int(model_validation_df["Exists"].sum())
n_missing = int((~model_validation_df["Exists"]).sum())

print(f"Models found         : {n_found}")
print(f"Models missing       : {n_missing}")
print()

if missing_models:
    print("MISSING MODELS:")
    for fold, k in missing_models[:20]:
        print(f"  Fold={fold}, K={k}")

    if len(missing_models) > 20:
        print(f"  ... and {len(missing_models) - 20} more")

    raise FileNotFoundError(
        f"{n_missing} expected LDA models are missing."
    )

print("✓ All expected LDA models are present.")

# ------------------------------------------------------------
# Fold-wise validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FOLD-WISE MODEL COUNTS")
print("=" * 70)

fold_counts = (
    model_validation_df
    .groupby("Fold")["Exists"]
    .sum()
)

for fold, count in fold_counts.items():
    print(f"Fold {fold}: {int(count)} models")

assert all(fold_counts == len(TOPIC_LIST))

# ------------------------------------------------------------
# K-wise validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("K RANGE VALIDATION")
print("=" * 70)

print(
    f"Minimum K found : "
    f"{model_validation_df.loc[model_validation_df['Exists'], 'Topics'].min()}"
)

print(
    f"Maximum K found : "
    f"{model_validation_df.loc[model_validation_df['Exists'], 'Topics'].max()}"
)

assert (
    model_validation_df["Topics"].min() == TOPIC_MIN
    and
    model_validation_df["Topics"].max() == TOPIC_MAX
)

print("\n" + "=" * 70)
print("VALIDATION PASSED")
print("=" * 70)

print(
    f"✓ {n_found}/{EXPECTED_EXPERIMENTS} "
    "LDA models are available."
)

In [ ]:
# ============================================================
# NG04-5 — Load NG03 Experiment Manifest
# ============================================================

MANIFEST_FILE = MODEL_DIR / "experiment_manifest_stage3.csv"

if not MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"NG03 manifest not found:\n{MANIFEST_FILE}"
    )

manifest_df = pd.read_csv(MANIFEST_FILE)

print("=" * 70)
print("NG03 EXPERIMENT MANIFEST")
print("=" * 70)

print("Shape:", manifest_df.shape)
print()

print("Columns:")
for col in manifest_df.columns:
    print("  ", col)

print("\nFirst 5 rows:")
display(manifest_df.head())

NG03 EXPERIMENT MANIFEST
Shape: (495, 10)

Columns:
   ExperimentID
   Fold
   Topics
   TrainDocuments
   TestDocuments
   DictionarySize
   ModelPath
   TrainVectorPath
   TestVectorPath
   Status

First 5 rows:


,ExperimentID,Fold,Topics,TrainDocuments,TestDocuments,DictionarySize,ModelPath,TrainVectorPath,TestVectorPath,Status
0,NG_F1_K2,1,2,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
1,NG_F1_K3,1,3,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
2,NG_F1_K4,1,4,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
3,NG_F1_K5,1,5,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
4,NG_F1_K6,1,6,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed


In [ ]:
# ============================================================
# NG04-5B — Validate Manifest
# ============================================================

print("=" * 70)
print("MANIFEST VALIDATION")
print("=" * 70)

print(f"Manifest rows : {len(manifest_df)}")
print(f"Expected rows : {EXPECTED_EXPERIMENTS}")

assert len(manifest_df) == EXPECTED_EXPERIMENTS, (
    f"Expected {EXPECTED_EXPERIMENTS} rows, "
    f"found {len(manifest_df)}."
)

print("✓ Manifest contains 495 experiments.")

# Check Fold/K uniqueness
possible_fold_k_columns = [
    ("Fold", "Topics"),
    ("fold", "topics"),
    ("Fold", "K"),
    ("fold", "K")
]

fold_col = None
topic_col = None

for fcol, tcol in possible_fold_k_columns:
    if fcol in manifest_df.columns and tcol in manifest_df.columns:
        fold_col = fcol
        topic_col = tcol
        break

if fold_col is None:
    raise KeyError(
        "Could not identify Fold and Topics/K columns in "
        "experiment_manifest_stage3.csv.\n"
        f"Available columns: {list(manifest_df.columns)}"
    )

print(f"Fold column   : {fold_col}")
print(f"Topic column  : {topic_col}")

duplicate_pairs = manifest_df.duplicated(
    subset=[fold_col, topic_col]
).sum()

print(f"Duplicate Fold/K pairs : {duplicate_pairs}")

assert duplicate_pairs == 0

print("✓ Fold/K combinations are unique.")

MANIFEST VALIDATION
Manifest rows : 495
Expected rows : 495
✓ Manifest contains 495 experiments.
Fold column   : Fold
Topic column  : Topics
Duplicate Fold/K pairs : 0
✓ Fold/K combinations are unique.


In [ ]:
# ============================================================
# NG04-6B — Validate Fold-Level Dictionaries
# ============================================================

DICTIONARY_DIR = BASE_DIR / "dictionary"

print("=" * 70)
print("NG03 FOLD-LEVEL DICTIONARY VALIDATION")
print("=" * 70)

print("Dictionary root:")
print(DICTIONARY_DIR)

assert DICTIONARY_DIR.exists(), \
    f"Dictionary directory not found: {DICTIONARY_DIR}"

dictionary_records = []
missing_folds = []

for fold in range(1, N_FOLDS + 1):

    fold_dir = DICTIONARY_DIR / f"Fold_{fold}"

    print(f"\n{'-' * 70}")
    print(f"FOLD {fold}")
    print(f"Directory: {fold_dir}")

    if not fold_dir.exists():
        print("✗ Fold dictionary directory does not exist.")
        missing_folds.append(fold)
        continue

    items = sorted(fold_dir.iterdir())

    if not items:
        print("✗ Directory is empty.")
        missing_folds.append(fold)
        continue

    print("Contents:")

    for item in items:
        print(" ", item.name)

        dictionary_records.append({
            "Fold": fold,
            "Path": str(item),
            "Name": item.name
        })

print("\n" + "=" * 70)
print("DICTIONARY VALIDATION SUMMARY")
print("=" * 70)

print(f"Expected folds : {N_FOLDS}")
print(f"Missing folds  : {missing_folds}")
print(f"Fold resources : {len(dictionary_records)}")

assert len(missing_folds) == 0, \
    f"Missing dictionary resources for folds: {missing_folds}"

print("\n✓ All five fold-level dictionary directories exist.")

NG03 FOLD-LEVEL DICTIONARY VALIDATION
Dictionary root:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary

----------------------------------------------------------------------
FOLD 1
Directory: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold_1
Contents:
  dictionary.dict

----------------------------------------------------------------------
FOLD 2
Directory: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold_2
Contents:
  dictionary.dict

----------------------------------------------------------------------
FOLD 3
Directory: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold_3
Contents:
  dictionary.dict

----------------------------------------------------------------------
FOLD 4
Directory: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold_4
Contents:
  dictionary.dict

----------------------------------------------------------------------
FOLD 5
Directory: /content/drive/MyDrive/LDAChat

In [ ]:
# ============================================================
# NG04-6C — Inspect Dictionary File Structure
# ============================================================

print("=" * 70)
print("DICTIONARY FILE STRUCTURE")
print("=" * 70)

for fold in range(1, N_FOLDS + 1):

    fold_dir = DICTIONARY_DIR / f"Fold_{fold}"

    print(f"\n{'-' * 70}")
    print(f"FOLD {fold}")

    for item in sorted(fold_dir.rglob("*")):

        if item.is_file():
            size_kb = item.stat().st_size / 1024

            print(
                f"  {item.relative_to(fold_dir)}"
                f"    ({size_kb:.2f} KB)"
            )

DICTIONARY FILE STRUCTURE

----------------------------------------------------------------------
FOLD 1
  dictionary.dict    (415.04 KB)

----------------------------------------------------------------------
FOLD 2
  dictionary.dict    (408.42 KB)

----------------------------------------------------------------------
FOLD 3
  dictionary.dict    (399.73 KB)

----------------------------------------------------------------------
FOLD 4
  dictionary.dict    (405.29 KB)

----------------------------------------------------------------------
FOLD 5
  dictionary.dict    (410.60 KB)


In [ ]:
# ============================================================
# NG04-7 — Validate Fold Dictionaries
# ============================================================

from gensim.corpora import Dictionary

print("=" * 70)
print("NG04 FOLD DICTIONARY VALIDATION")
print("=" * 70)

DICTIONARY_DIR = BASE_DIR / "dictionary"

dictionary_info = []

for fold in range(1, N_FOLDS + 1):

    dictionary_path = (
        DICTIONARY_DIR
        / f"Fold_{fold}"
        / "dictionary.dict"
    )

    print(f"\n{'-' * 70}")
    print(f"FOLD {fold}")
    print("Dictionary:", dictionary_path)

    assert dictionary_path.exists(), (
        f"Dictionary not found: {dictionary_path}"
    )

    dictionary = Dictionary.load(str(dictionary_path))

    vocab_size = len(dictionary)

    print("Vocabulary size:", vocab_size)

    dictionary_info.append({
        "Fold": fold,
        "DictionaryPath": str(dictionary_path),
        "DictionarySize": vocab_size
    })

dictionary_df = pd.DataFrame(dictionary_info)

print("\n" + "=" * 70)
print("DICTIONARY SUMMARY")
print("=" * 70)

display(dictionary_df)

NG04 FOLD DICTIONARY VALIDATION

----------------------------------------------------------------------
FOLD 1
Dictionary: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold_1/dictionary.dict
Vocabulary size: 18264

----------------------------------------------------------------------
FOLD 2
Dictionary: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold_2/dictionary.dict
Vocabulary size: 17976

----------------------------------------------------------------------
FOLD 3
Dictionary: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold_3/dictionary.dict
Vocabulary size: 17589

----------------------------------------------------------------------
FOLD 4
Dictionary: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold_4/dictionary.dict
Vocabulary size: 17883

----------------------------------------------------------------------
FOLD 5
Dictionary: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/dictionary/Fold

,Fold,DictionaryPath,DictionarySize
0,1,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,18264
1,2,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,17976
2,3,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,17589
3,4,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,17883
4,5,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,18083


In [ ]:
# ============================================================
# NG04-7B — Compare Manifest Dictionary Size
#              with Actual Dictionary Size
# ============================================================

print("=" * 70)
print("MANIFEST vs ACTUAL DICTIONARY SIZE")
print("=" * 70)

manifest_dictionary_sizes = (
    manifest_df
    .groupby("Fold")["DictionarySize"]
    .first()
    .reset_index()
)

manifest_dictionary_sizes = manifest_dictionary_sizes.rename(
    columns={
        "DictionarySize": "ManifestDictionarySize"
    }
)

dictionary_check = manifest_dictionary_sizes.merge(
    dictionary_df[["Fold", "DictionarySize"]],
    on="Fold",
    how="left"
)

dictionary_check["Match"] = (
    dictionary_check["ManifestDictionarySize"]
    == dictionary_check["DictionarySize"]
)

display(dictionary_check)

print("\n" + "=" * 70)
print("VALIDATION RESULT")
print("=" * 70)

for _, row in dictionary_check.iterrows():
    print(
        f"Fold {int(row['Fold'])}: "
        f"Manifest={int(row['ManifestDictionarySize'])}, "
        f"Actual={int(row['DictionarySize'])}, "
        f"Match={row['Match']}"
    )

assert dictionary_check["Match"].all(), (
    "Dictionary size mismatch detected."
)

print("\n✓ All fold dictionaries match the NG03 manifest.")

MANIFEST vs ACTUAL DICTIONARY SIZE


,Fold,ManifestDictionarySize,DictionarySize,Match
0,1,18264,18264,True
1,2,17976,17976,True
2,3,17589,17589,True
3,4,17883,17883,True
4,5,18083,18083,True



VALIDATION RESULT
Fold 1: Manifest=18264, Actual=18264, Match=True
Fold 2: Manifest=17976, Actual=17976, Match=True
Fold 3: Manifest=17589, Actual=17589, Match=True
Fold 4: Manifest=17883, Actual=17883, Match=True
Fold 5: Manifest=18083, Actual=18083, Match=True

✓ All fold dictionaries match the NG03 manifest.


In [ ]:
# --------------------------------------------------
# 8A
# --------------------------------------------------
from pathlib import Path
import pandas as pd
import numpy as np
from gensim.models import LdaModel
from gensim.corpora import Dictionary

BASE_DIR = Path(
    "/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval"
)

MODEL_DIR = BASE_DIR / "models"
DICTIONARY_DIR = BASE_DIR / "dictionary"
ANALYSIS_DIR = BASE_DIR / "analysis"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = MODEL_DIR / "experiment_manifest_stage3.csv"

manifest_df = pd.read_csv(MANIFEST_FILE)

print("Manifest rows:", len(manifest_df))
print("Expected rows:", 5 * 99)

assert len(manifest_df) == 495
assert manifest_df[["Fold", "Topics"]].drop_duplicates().shape[0] == 495

print("✓ NG03 manifest loaded and validated.")

Manifest rows: 495
Expected rows: 495
✓ NG03 manifest loaded and validated.


In [ ]:
# --------------------------------------------------
# 8B
# --------------------------------------------------
print("Manifest columns:")
print(manifest_df.columns.tolist())

print("\nFirst five rows:")
display(manifest_df.head())

Manifest columns:
['ExperimentID', 'Fold', 'Topics', 'TrainDocuments', 'TestDocuments', 'DictionarySize', 'ModelPath', 'TrainVectorPath', 'TestVectorPath', 'Status']

First five rows:


,ExperimentID,Fold,Topics,TrainDocuments,TestDocuments,DictionarySize,ModelPath,TrainVectorPath,TestVectorPath,Status
0,NG_F1_K2,1,2,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
1,NG_F1_K3,1,3,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
2,NG_F1_K4,1,4,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
3,NG_F1_K5,1,5,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed
4,NG_F1_K6,1,6,14637,3660,18264,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,/content/drive/MyDrive/LDAChatGPT/NewsGroupsTo...,Completed


In [ ]:
# ============================================================
# NG04-8M — Reconstruct Exact NG03 Train/Test BOW Corpora
# ============================================================

import ast
import pandas as pd
from pathlib import Path
from gensim.corpora import Dictionary

print("=" * 80)
print("NG04-8M — RECONSTRUCT EXACT NG03 CORPORA")
print("=" * 80)

# ------------------------------------------------------------
# Token restoration — copied from NG03
# ------------------------------------------------------------

def restore_tokens(value):

    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    if isinstance(value, str):

        try:
            tokens = ast.literal_eval(value)

            if isinstance(tokens, list):
                return [
                    str(token)
                    for token in tokens
                ]

        except Exception:
            pass

        return value.split()

    return []


# ------------------------------------------------------------
# Load fold exactly as NG03 did
# ------------------------------------------------------------

def load_ng03_fold(fold):

    train_file = (
        BASE_DIR
        / "folds"
        / f"Fold_{fold}"
        / "train.xlsx"
    )

    test_file = (
        BASE_DIR
        / "folds"
        / f"Fold_{fold}"
        / "test.xlsx"
    )

    train = pd.read_excel(train_file)
    test = pd.read_excel(test_file)

    assert "Tokens" in train.columns
    assert "Tokens" in test.columns

    train["Tokens"] = (
        train["Tokens"]
        .apply(restore_tokens)
    )

    test["Tokens"] = (
        test["Tokens"]
        .apply(restore_tokens)
    )

    return train, test


# ------------------------------------------------------------
# Reconstruct exact fold resources
# ------------------------------------------------------------

ng04_fold_resources = {}

validation_rows = []

for fold in range(1, 6):

    print("\n" + "=" * 70)
    print(f"Fold {fold}")
    print("=" * 70)

    train_df, test_df = load_ng03_fold(fold)

    train_tokens = train_df["Tokens"].tolist()
    test_tokens = test_df["Tokens"].tolist()

    # --------------------------------------------------------
    # Load SAME dictionary used by trained models
    # --------------------------------------------------------

    dictionary_path = (
        BASE_DIR
        / "dictionary"
        / f"Fold_{fold}"
        / "dictionary.dict"
    )

    dictionary = Dictionary.load(
        str(dictionary_path)
    )

    # --------------------------------------------------------
    # Exact BOW transformation used in NG03
    # --------------------------------------------------------

    train_corpus = [
        dictionary.doc2bow(document)
        for document in train_tokens
    ]

    test_corpus = [
        dictionary.doc2bow(document)
        for document in test_tokens
    ]

    # --------------------------------------------------------
    # Manifest reference
    # --------------------------------------------------------

    mf = (
        manifest_df[
            manifest_df["Fold"] == fold
        ]
        .iloc[0]
    )

    expected_train = int(
        mf["TrainDocuments"]
    )

    expected_test = int(
        mf["TestDocuments"]
    )

    expected_dict = int(
        mf["DictionarySize"]
    )

    print(
        "Train documents :",
        len(train_corpus),
        "| expected:",
        expected_train
    )

    print(
        "Test documents  :",
        len(test_corpus),
        "| expected:",
        expected_test
    )

    print(
        "Dictionary size :",
        len(dictionary),
        "| expected:",
        expected_dict
    )

    train_match = (
        len(train_corpus)
        == expected_train
    )

    test_match = (
        len(test_corpus)
        == expected_test
    )

    dict_match = (
        len(dictionary)
        == expected_dict
    )

    print()
    print("Train count match :", train_match)
    print("Test count match  :", test_match)
    print("Dictionary match  :", dict_match)

    assert train_match
    assert test_match
    assert dict_match

    # --------------------------------------------------------
    # Store for NG04 metrics
    # --------------------------------------------------------

    ng04_fold_resources[fold] = {

        "TrainDataFrame":
            train_df,

        "TestDataFrame":
            test_df,

        "TrainTokens":
            train_tokens,

        "TestTokens":
            test_tokens,

        "Dictionary":
            dictionary,

        "TrainCorpus":
            train_corpus,

        "TestCorpus":
            test_corpus
    }

    validation_rows.append({

        "Fold": fold,

        "TrainDocuments":
            len(train_corpus),

        "ExpectedTrain":
            expected_train,

        "TestDocuments":
            len(test_corpus),

        "ExpectedTest":
            expected_test,

        "DictionarySize":
            len(dictionary),

        "ExpectedDictionary":
            expected_dict,

        "Valid":
            (
                train_match
                and test_match
                and dict_match
            )
    })


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

corpus_validation_df = pd.DataFrame(
    validation_rows
)

print("\n" + "=" * 80)
print("CORPUS RECONSTRUCTION SUMMARY")
print("=" * 80)

print(
    corpus_validation_df.to_string(
        index=False
    )
)

assert corpus_validation_df["Valid"].all()

print("\n" + "=" * 80)
print("✓ EXACT NG03 CORPORA SUCCESSFULLY RECONSTRUCTED")
print("✓ READY FOR PERPLEXITY / COHERENCE COMPUTATION")
print("=" * 80)

NG04-8M — RECONSTRUCT EXACT NG03 CORPORA

Fold 1
Train documents : 14637 | expected: 14637
Test documents  : 3660 | expected: 3660
Dictionary size : 18264 | expected: 18264

Train count match : True
Test count match  : True
Dictionary match  : True

Fold 2
Train documents : 14637 | expected: 14637
Test documents  : 3660 | expected: 3660
Dictionary size : 17976 | expected: 17976

Train count match : True
Test count match  : True
Dictionary match  : True

Fold 3
Train documents : 14638 | expected: 14638
Test documents  : 3659 | expected: 3659
Dictionary size : 17589 | expected: 17589

Train count match : True
Test count match  : True
Dictionary match  : True

Fold 4
Train documents : 14638 | expected: 14638
Test documents  : 3659 | expected: 3659
Dictionary size : 17883 | expected: 17883

Train count match : True
Test count match  : True
Dictionary match  : True

Fold 5
Train documents : 14638 | expected: 14638
Test documents  : 3659 | expected: 3659
Dictionary size : 18083 | expected: 1

In [ ]:
# ============================================================
# NG04-8N — Single-Model Perplexity Validation
# ============================================================

from gensim.models import LdaModel
import numpy as np

print("=" * 80)
print("NG04-8N — SINGLE-MODEL PERPLEXITY VALIDATION")
print("=" * 80)

VALIDATION_FOLD = 1
VALIDATION_K = 2

# ------------------------------------------------------------
# Load exact reconstructed corpora
# ------------------------------------------------------------

resources = ng04_fold_resources[VALIDATION_FOLD]

train_corpus = resources["TrainCorpus"]
test_corpus  = resources["TestCorpus"]

# ------------------------------------------------------------
# Load saved NG03 model
# ------------------------------------------------------------

model_path = (
    BASE_DIR
    / "models"
    / f"Fold_{VALIDATION_FOLD}"
    / f"K_{VALIDATION_K}"
    / "lda_model"
)

model = LdaModel.load(str(model_path))

print("\nModel:")
print(model_path)

print("\nFold:", VALIDATION_FOLD)
print("K   :", VALIDATION_K)

print(
    "Train documents:",
    len(train_corpus)
)

print(
    "Test documents :",
    len(test_corpus)
)

# ------------------------------------------------------------
# Gensim log perplexity
# ------------------------------------------------------------

train_log_perplexity = model.log_perplexity(
    train_corpus
)

test_log_perplexity = model.log_perplexity(
    test_corpus
)

# Gensim's corresponding conventional perplexity
train_perplexity = 2 ** (
    -train_log_perplexity
)

test_perplexity = 2 ** (
    -test_log_perplexity
)

# ------------------------------------------------------------
# Validate numerical output
# ------------------------------------------------------------

assert np.isfinite(train_log_perplexity)
assert np.isfinite(test_log_perplexity)

assert train_perplexity > 0
assert test_perplexity > 0

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\n" + "-" * 60)

print(
    f"Train log-perplexity : "
    f"{train_log_perplexity:.8f}"
)

print(
    f"Test log-perplexity  : "
    f"{test_log_perplexity:.8f}"
)

print(
    f"Train perplexity     : "
    f"{train_perplexity:.8f}"
)

print(
    f"Test perplexity      : "
    f"{test_perplexity:.8f}"
)

print("-" * 60)

print("\n✓ Perplexity computation successful.")
print("✓ Exact NG03 BOW corpus accepted by saved LDA model.")
print("✓ Ready for the 495-model metric computation.")

NG04-8N — SINGLE-MODEL PERPLEXITY VALIDATION

Model:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/models/Fold_1/K_2/lda_model

Fold: 1
K   : 2
Train documents: 14637
Test documents : 3660

------------------------------------------------------------
Train log-perplexity : -8.16694155
Test log-perplexity  : -8.21858425
Train perplexity     : 287.40503889
Test perplexity      : 297.87933826
------------------------------------------------------------

✓ Perplexity computation successful.
✓ Exact NG03 BOW corpus accepted by saved LDA model.
✓ Ready for the 495-model metric computation.


In [ ]:
# ============================================================
# NG04-8O — Compute Perplexity for All 495 LDA Models
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from pathlib import Path
from gensim.models import LdaModel

print("=" * 80)
print("NG04-8O — PERPLEXITY FOR ALL 495 MODELS")
print("=" * 80)

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

METRIC_DIR = BASE_DIR / "metrics"
METRIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PERPLEXITY_CHECKPOINT = (
    METRIC_DIR
    / "ng04_perplexity_checkpoint.csv"
)

PERPLEXITY_FINAL = (
    METRIC_DIR
    / "ng04_perplexity_all_models.csv"
)

# ------------------------------------------------------------
# Load existing checkpoint if available
# ------------------------------------------------------------

if PERPLEXITY_CHECKPOINT.exists():

    perplexity_df = pd.read_csv(
        PERPLEXITY_CHECKPOINT
    )

    print(
        "\nExisting checkpoint found:",
        len(perplexity_df),
        "experiments"
    )

else:

    perplexity_df = pd.DataFrame(
        columns=[
            "ExperimentID",
            "Fold",
            "Topics",
            "TrainLogPerplexity",
            "TestLogPerplexity",
            "TrainPerplexity",
            "TestPerplexity"
        ]
    )

    print(
        "\nNo checkpoint found."
    )


# ------------------------------------------------------------
# Completed experiments
# ------------------------------------------------------------

if len(perplexity_df) > 0:

    completed = set(
        zip(
            perplexity_df["Fold"].astype(int),
            perplexity_df["Topics"].astype(int)
        )
    )

else:

    completed = set()


print(
    "Already completed:",
    len(completed)
)

print(
    "Remaining:",
    495 - len(completed)
)


# ------------------------------------------------------------
# Timer
# ------------------------------------------------------------

start_time = time.time()

new_count = 0


# ============================================================
# Process folds
# ============================================================

for fold in range(1, 6):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}")
    print("=" * 70)

    resources = ng04_fold_resources[fold]

    train_corpus = resources[
        "TrainCorpus"
    ]

    test_corpus = resources[
        "TestCorpus"
    ]

    # --------------------------------------------------------
    # Process K = 2 ... 100
    # --------------------------------------------------------

    for topics in range(2, 101):

        if (fold, topics) in completed:

            print(
                f"Fold={fold}, K={topics}: "
                f"already completed — skipping"
            )

            continue

        experiment_id = (
            f"NG_F{fold}_K{topics}"
        )

        model_path = (
            BASE_DIR
            / "models"
            / f"Fold_{fold}"
            / f"K_{topics}"
            / "lda_model"
        )

        try:

            # ------------------------------------------------
            # Load trained NG03 model
            # ------------------------------------------------

            model = LdaModel.load(
                str(model_path)
            )

            # ------------------------------------------------
            # Log-perplexity
            # ------------------------------------------------

            train_log = float(
                model.log_perplexity(
                    train_corpus
                )
            )

            test_log = float(
                model.log_perplexity(
                    test_corpus
                )
            )

            # ------------------------------------------------
            # Conventional perplexity
            # ------------------------------------------------

            train_perp = float(
                2 ** (-train_log)
            )

            test_perp = float(
                2 ** (-test_log)
            )

            # ------------------------------------------------
            # Numerical validation
            # ------------------------------------------------

            assert np.isfinite(train_log)
            assert np.isfinite(test_log)

            assert np.isfinite(train_perp)
            assert np.isfinite(test_perp)

            assert train_perp > 0
            assert test_perp > 0

            # ------------------------------------------------
            # Result
            # ------------------------------------------------

            result_row = {
                "ExperimentID":
                    experiment_id,

                "Fold":
                    fold,

                "Topics":
                    topics,

                "TrainLogPerplexity":
                    train_log,

                "TestLogPerplexity":
                    test_log,

                "TrainPerplexity":
                    train_perp,

                "TestPerplexity":
                    test_perp
            }

            perplexity_df = pd.concat(
                [
                    perplexity_df,
                    pd.DataFrame(
                        [result_row]
                    )
                ],
                ignore_index=True
            )

            # ------------------------------------------------
            # Save immediately
            # ------------------------------------------------

            perplexity_df.to_csv(
                PERPLEXITY_CHECKPOINT,
                index=False
            )

            completed.add(
                (fold, topics)
            )

            new_count += 1

            print(
                f"✓ Fold={fold}, K={topics} | "
                f"TrainLog={train_log:.5f} | "
                f"TestLog={test_log:.5f} | "
                f"TestPerp={test_perp:.3f}"
            )

            # ------------------------------------------------
            # Memory cleanup
            # ------------------------------------------------

            del model

            gc.collect()

        except Exception as e:

            print(
                f"✗ ERROR Fold={fold}, "
                f"K={topics}: {e}"
            )

            # Preserve everything obtained so far
            perplexity_df.to_csv(
                PERPLEXITY_CHECKPOINT,
                index=False
            )

            gc.collect()


# ============================================================
# Sort results
# ============================================================

perplexity_df["Fold"] = (
    perplexity_df["Fold"]
    .astype(int)
)

perplexity_df["Topics"] = (
    perplexity_df["Topics"]
    .astype(int)
)

perplexity_df = (
    perplexity_df
    .sort_values(
        ["Fold", "Topics"]
    )
    .drop_duplicates(
        subset=[
            "Fold",
            "Topics"
        ],
        keep="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Save final result
# ------------------------------------------------------------

perplexity_df.to_csv(
    PERPLEXITY_FINAL,
    index=False
)

perplexity_df.to_csv(
    PERPLEXITY_CHECKPOINT,
    index=False
)


# ============================================================
# Validation
# ============================================================

elapsed = time.time() - start_time

expected = 5 * 99

print("\n" + "=" * 80)
print("PERPLEXITY COMPUTATION SUMMARY")
print("=" * 80)

print(
    "Expected experiments :",
    expected
)

print(
    "Completed experiments:",
    len(perplexity_df)
)

print(
    "Newly computed       :",
    new_count
)

print(
    "Duplicate Fold-K     :",
    perplexity_df.duplicated(
        ["Fold", "Topics"]
    ).sum()
)

print(
    "Elapsed minutes      :",
    round(elapsed / 60, 2)
)

print(
    "\nFinal file:"
)

print(
    PERPLEXITY_FINAL
)


# ------------------------------------------------------------
# Strict final checks
# ------------------------------------------------------------

assert len(perplexity_df) == expected, (
    f"Expected {expected} results, "
    f"found {len(perplexity_df)}."
)

assert (
    perplexity_df
    .duplicated(
        ["Fold", "Topics"]
    )
    .sum()
    == 0
)

assert np.isfinite(
    perplexity_df[
        [
            "TrainLogPerplexity",
            "TestLogPerplexity",
            "TrainPerplexity",
            "TestPerplexity"
        ]
    ].to_numpy()
).all()


print("\n" + "=" * 80)

print(
    "✓ ALL 495 PERPLEXITY RESULTS COMPUTED"
)

print(
    "✓ CHECKPOINT SAVED"
)

print(
    "✓ FINAL PERPLEXITY TABLE SAVED"
)

print("=" * 80)


display(
    perplexity_df.head(10)
)

NG04-8O — PERPLEXITY FOR ALL 495 MODELS

Existing checkpoint found: 487 experiments
Already completed: 487
Remaining: 8

FOLD 1
Fold=1, K=2: already completed — skipping
Fold=1, K=3: already completed — skipping
Fold=1, K=4: already completed — skipping
Fold=1, K=5: already completed — skipping
Fold=1, K=6: already completed — skipping
Fold=1, K=7: already completed — skipping
Fold=1, K=8: already completed — skipping
Fold=1, K=9: already completed — skipping
Fold=1, K=10: already completed — skipping
Fold=1, K=11: already completed — skipping
Fold=1, K=12: already completed — skipping
Fold=1, K=13: already completed — skipping
Fold=1, K=14: already completed — skipping
Fold=1, K=15: already completed — skipping
Fold=1, K=16: already completed — skipping
Fold=1, K=17: already completed — skipping
Fold=1, K=18: already completed — skipping
Fold=1, K=19: already completed — skipping
Fold=1, K=20: already completed — skipping
Fold=1, K=21: already completed — skipping
Fold=1, K=22: alread

,ExperimentID,Fold,Topics,TrainLogPerplexity,TestLogPerplexity,TrainPerplexity,TestPerplexity
0,NG_F1_K2,1,2,-8.166942,-8.218584,287.405039,297.879338
1,NG_F1_K3,1,3,-8.017795,-8.114469,259.177277,277.139657
2,NG_F1_K4,1,4,-7.962156,-8.111846,249.372131,276.636202
3,NG_F1_K5,1,5,-7.929313,-8.109783,243.759208,276.240885
4,NG_F1_K6,1,6,-7.908885,-8.111695,240.331941,276.607296
5,NG_F1_K7,1,7,-7.927259,-8.142549,243.412343,282.586535
6,NG_F1_K8,1,8,-7.950640,-8.171427,247.389346,288.299923
7,NG_F1_K9,1,9,-7.949942,-8.178067,247.269690,289.630044
8,NG_F1_K10,1,10,-7.989076,-8.215569,254.068875,297.257459
9,NG_F1_K11,1,11,-8.092732,-8.323661,272.995229,320.384566


In [ ]:
# ============================================================
# NG04-8P — Final Perplexity Result Summary
# ============================================================

METRIC_DIR = BASE_DIR / "metrics"
METRIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PERPLEXITY_CHECKPOINT = (
    METRIC_DIR
    / "ng04_perplexity_checkpoint.csv"
)

PERPLEXITY_FINAL = (
    METRIC_DIR
    / "ng04_perplexity_all_models.csv"
)


print("=" * 80)
print("NG04-8P — FINAL PERPLEXITY RESULT SUMMARY")
print("=" * 80)

perplexity_df = pd.read_csv(
    PERPLEXITY_FINAL
)

print("Shape              :", perplexity_df.shape)

print(
    "Unique Fold-K pairs:",
    perplexity_df[
        ["Fold", "Topics"]
    ].drop_duplicates().shape[0]
)

print(
    "Missing values     :",
    perplexity_df.isna().sum().sum()
)

print()

summary = (
    perplexity_df
    .groupby("Fold")
    .agg(
        Models=("Topics", "count"),
        MinK=("Topics", "min"),
        MaxK=("Topics", "max"),
        MinTestLogPerplexity=(
            "TestLogPerplexity",
            "min"
        ),
        MaxTestLogPerplexity=(
            "TestLogPerplexity",
            "max"
        ),
        MinTestPerplexity=(
            "TestPerplexity",
            "min"
        ),
        MaxTestPerplexity=(
            "TestPerplexity",
            "max"
        )
    )
)

display(summary)

# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

assert len(perplexity_df) == 495

assert (
    perplexity_df[
        ["Fold", "Topics"]
    ]
    .drop_duplicates()
    .shape[0]
    == 495
)

assert perplexity_df.isna().sum().sum() == 0

assert set(
    perplexity_df["Fold"].unique()
) == {1, 2, 3, 4, 5}

for fold in range(1, 6):

    ks = set(
        perplexity_df.loc[
            perplexity_df["Fold"] == fold,
            "Topics"
        ]
    )

    assert ks == set(
        range(2, 101)
    )

print()
print("=" * 80)
print("✓ 495 UNIQUE PERPLEXITY RESULTS VERIFIED")
print("✓ ALL FOLDS CONTAIN K = 2 ... 100")
print("✓ NO MISSING VALUES")
print("=" * 80)

NG04-8P — FINAL PERPLEXITY RESULT SUMMARY
Shape              : (495, 7)
Unique Fold-K pairs: 495
Missing values     : 0



,Models,MinK,MaxK,MinTestLogPerplexity,MaxTestLogPerplexity,MinTestPerplexity,MaxTestPerplexity
Fold,,,,,,,
1,99,2,100,-15.484621,-8.109783,276.240885,45849.595997
2,99,2,100,-11.452576,-8.248373,304.093850,2802.649939
3,99,2,100,-12.210208,-7.896106,238.212537,4738.479128
4,99,2,100,-11.562086,-8.268954,308.463046,3023.671645
5,99,2,100,-11.861954,-8.315509,318.579367,3722.237205



✓ 495 UNIQUE PERPLEXITY RESULTS VERIFIED
✓ ALL FOLDS CONTAIN K = 2 ... 100
✓ NO MISSING VALUES


In [ ]:
# ============================================================
# NG04-8Q — Compute Four Coherence Metrics for All 495 Models
# ============================================================

import gc
import time
import numpy as np
import pandas as pd

from pathlib import Path
from gensim.models import LdaModel, CoherenceModel

print("=" * 80)
print("NG04-8Q — COHERENCE METRICS FOR ALL 495 MODELS")
print("=" * 80)

# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

METRIC_DIR = BASE_DIR / "metrics"

METRIC_DIR.mkdir(
    parents=True,
    exist_ok=True
)

COHERENCE_CHECKPOINT = (
    METRIC_DIR
    / "ng04_coherence_checkpoint.csv"
)

COHERENCE_FINAL = (
    METRIC_DIR
    / "ng04_coherence_all_models.csv"
)

# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

if COHERENCE_CHECKPOINT.exists():

    coherence_df = pd.read_csv(
        COHERENCE_CHECKPOINT
    )

    print(
        "\nExisting checkpoint found:",
        len(coherence_df),
        "experiments"
    )

else:

    coherence_df = pd.DataFrame(
        columns=[
            "ExperimentID",
            "Fold",
            "Topics",
            "Coherence_UMass",
            "Coherence_CV",
            "Coherence_UCI",
            "Coherence_NPMI"
        ]
    )

    print("\nNo checkpoint found.")


# ------------------------------------------------------------
# Completed Fold-K pairs
# ------------------------------------------------------------

if len(coherence_df) > 0:

    completed = set(
        zip(
            coherence_df["Fold"].astype(int),
            coherence_df["Topics"].astype(int)
        )
    )

else:

    completed = set()


print(
    "Already completed:",
    len(completed)
)

print(
    "Remaining:",
    495 - len(completed)
)


# ------------------------------------------------------------
# Timer
# ------------------------------------------------------------

start_time = time.time()

new_count = 0


# ============================================================
# Fold loop
# ============================================================

for fold in range(1, 6):

    print("\n" + "=" * 70)
    print(f"FOLD {fold}")
    print("=" * 70)

    resources = ng04_fold_resources[fold]

    train_tokens = resources[
        "TrainTokens"
    ]

    train_corpus = resources[
        "TrainCorpus"
    ]

    dictionary = resources[
        "Dictionary"
    ]

    print(
        "Training documents:",
        len(train_tokens)
    )

    print(
        "Dictionary size   :",
        len(dictionary)
    )

    # ========================================================
    # K loop
    # ========================================================

    for topics in range(2, 101):

        if (fold, topics) in completed:

            print(
                f"Fold={fold}, K={topics}: "
                "already completed — skipping"
            )

            continue

        experiment_id = (
            f"NG_F{fold}_K{topics}"
        )

        model_path = (
            BASE_DIR
            / "models"
            / f"Fold_{fold}"
            / f"K_{topics}"
            / "lda_model"
        )

        try:

            # ------------------------------------------------
            # Load existing NG03 model
            # ------------------------------------------------

            model = LdaModel.load(
                str(model_path)
            )

            # =================================================
            # 1. UMass
            # =================================================

            cm_umass = CoherenceModel(
                model=model,
                corpus=train_corpus,
                dictionary=dictionary,
                coherence="u_mass"
            )

            coherence_umass = float(
                cm_umass.get_coherence()
            )

            del cm_umass

            # =================================================
            # 2. C_V
            # =================================================

            cm_cv = CoherenceModel(
                model=model,
                texts=train_tokens,
                dictionary=dictionary,
                coherence="c_v",
                processes=1
            )

            coherence_cv = float(
                cm_cv.get_coherence()
            )

            del cm_cv

            # =================================================
            # 3. C_UCI
            # =================================================

            cm_uci = CoherenceModel(
                model=model,
                texts=train_tokens,
                dictionary=dictionary,
                coherence="c_uci",
                processes=1
            )

            coherence_uci = float(
                cm_uci.get_coherence()
            )

            del cm_uci

            # =================================================
            # 4. C_NPMI
            # =================================================

            cm_npmi = CoherenceModel(
                model=model,
                texts=train_tokens,
                dictionary=dictionary,
                coherence="c_npmi",
                processes=1
            )

            coherence_npmi = float(
                cm_npmi.get_coherence()
            )

            del cm_npmi

            # ------------------------------------------------
            # Numerical validation
            # ------------------------------------------------

            values = np.array(
                [
                    coherence_umass,
                    coherence_cv,
                    coherence_uci,
                    coherence_npmi
                ],
                dtype=float
            )

            assert np.isfinite(
                values
            ).all()

            # ------------------------------------------------
            # Save result
            # ------------------------------------------------

            result_row = {

                "ExperimentID":
                    experiment_id,

                "Fold":
                    fold,

                "Topics":
                    topics,

                "Coherence_UMass":
                    coherence_umass,

                "Coherence_CV":
                    coherence_cv,

                "Coherence_UCI":
                    coherence_uci,

                "Coherence_NPMI":
                    coherence_npmi
            }

            coherence_df = pd.concat(
                [
                    coherence_df,
                    pd.DataFrame(
                        [result_row]
                    )
                ],
                ignore_index=True
            )

            # ------------------------------------------------
            # Checkpoint immediately
            # ------------------------------------------------

            coherence_df.to_csv(
                COHERENCE_CHECKPOINT,
                index=False
            )

            completed.add(
                (fold, topics)
            )

            new_count += 1

            print(
                f"✓ Fold={fold}, K={topics} | "
                f"UMass={coherence_umass:.5f} | "
                f"CV={coherence_cv:.5f} | "
                f"UCI={coherence_uci:.5f} | "
                f"NPMI={coherence_npmi:.5f}"
            )

            # ------------------------------------------------
            # Cleanup
            # ------------------------------------------------

            del model

            gc.collect()

        except Exception as e:

            print(
                f"✗ ERROR Fold={fold}, "
                f"K={topics}: {e}"
            )

            coherence_df.to_csv(
                COHERENCE_CHECKPOINT,
                index=False
            )

            gc.collect()


# ============================================================
# Final cleanup / sorting
# ============================================================

coherence_df["Fold"] = (
    coherence_df["Fold"]
    .astype(int)
)

coherence_df["Topics"] = (
    coherence_df["Topics"]
    .astype(int)
)

coherence_df = (
    coherence_df
    .sort_values(
        ["Fold", "Topics"]
    )
    .drop_duplicates(
        subset=[
            "Fold",
            "Topics"
        ],
        keep="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Save final table
# ------------------------------------------------------------

coherence_df.to_csv(
    COHERENCE_FINAL,
    index=False
)

coherence_df.to_csv(
    COHERENCE_CHECKPOINT,
    index=False
)


# ============================================================
# Final validation
# ============================================================

expected = 5 * 99

elapsed = (
    time.time()
    - start_time
)

print("\n" + "=" * 80)
print("COHERENCE COMPUTATION SUMMARY")
print("=" * 80)

print(
    "Expected experiments :",
    expected
)

print(
    "Completed experiments:",
    len(coherence_df)
)

print(
    "Newly computed       :",
    new_count
)

print(
    "Duplicate Fold-K     :",
    coherence_df.duplicated(
        ["Fold", "Topics"]
    ).sum()
)

print(
    "Elapsed minutes      :",
    round(
        elapsed / 60,
        2
    )
)

print(
    "\nFinal file:"
)

print(
    COHERENCE_FINAL
)


# ------------------------------------------------------------
# Strict checks
# ------------------------------------------------------------

assert len(coherence_df) == 495

assert (
    coherence_df
    .duplicated(
        ["Fold", "Topics"]
    )
    .sum()
    == 0
)

metric_columns = [
    "Coherence_UMass",
    "Coherence_CV",
    "Coherence_UCI",
    "Coherence_NPMI"
]

assert np.isfinite(
    coherence_df[
        metric_columns
    ].to_numpy()
).all()


print("\n" + "=" * 80)

print(
    "✓ ALL 495 COHERENCE RESULTS COMPUTED"
)

print(
    "✓ FOUR COHERENCE METRICS COMPLETE"
)

print(
    "✓ CHECKPOINT AND FINAL TABLE SAVED"
)

print("=" * 80)

display(
    coherence_df.head(10)
)

NG04-8Q — COHERENCE METRICS FOR ALL 495 MODELS

Existing checkpoint found: 188 experiments
Already completed: 188
Remaining: 307

FOLD 1
Training documents: 14637
Dictionary size   : 18264
Fold=1, K=2: already completed — skipping
Fold=1, K=3: already completed — skipping
Fold=1, K=4: already completed — skipping
Fold=1, K=5: already completed — skipping
Fold=1, K=6: already completed — skipping
Fold=1, K=7: already completed — skipping
Fold=1, K=8: already completed — skipping
Fold=1, K=9: already completed — skipping
Fold=1, K=10: already completed — skipping
Fold=1, K=11: already completed — skipping
Fold=1, K=12: already completed — skipping
Fold=1, K=13: already completed — skipping
Fold=1, K=14: already completed — skipping
Fold=1, K=15: already completed — skipping
Fold=1, K=16: already completed — skipping
Fold=1, K=17: already completed — skipping
Fold=1, K=18: already completed — skipping
Fold=1, K=19: already completed — skipping
Fold=1, K=20: already completed — skipping
Fo

,ExperimentID,Fold,Topics,Coherence_UMass,Coherence_CV,Coherence_UCI,Coherence_NPMI
0,NG_F1_K2,1,2,-6.624327,0.482003,-3.988233,-0.077489
1,NG_F1_K3,1,3,-1.259477,0.679111,0.346130,0.106190
2,NG_F1_K4,1,4,-1.392020,0.674515,0.317378,0.092768
3,NG_F1_K5,1,5,-1.521834,0.713990,0.531896,0.112504
4,NG_F1_K6,1,6,-1.502607,0.711919,0.554199,0.104472
5,NG_F1_K7,1,7,-1.848268,0.661607,0.029384,0.077049
6,NG_F1_K8,1,8,-2.852878,0.627860,-0.894749,0.038876
7,NG_F1_K9,1,9,-1.645164,0.691442,0.091416,0.089063
8,NG_F1_K10,1,10,-1.984278,0.685445,-0.150716,0.077739
9,NG_F1_K11,1,11,-1.840151,0.702560,0.168077,0.090852


In [ ]:
# ============================================================
# NG04-8R — Validate Raw Metrics and Build Master Metric Table
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("NG04-8R — RAW METRIC VALIDATION AND MERGE")
print("=" * 80)

# ------------------------------------------------------------
# Reload final files
# ------------------------------------------------------------


METRIC_DIR = BASE_DIR / "metrics"

COHERENCE_FINAL = (
    METRIC_DIR
    / "ng04_coherence_all_models.csv"
)

PERPLEXITY_FINAL = (
    METRIC_DIR
    / "ng04_perplexity_all_models.csv"
)


perplexity_df = pd.read_csv(
    PERPLEXITY_FINAL
)

coherence_df = pd.read_csv(
    COHERENCE_FINAL
)

# ------------------------------------------------------------
# Basic integrity
# ------------------------------------------------------------

print("\nPerplexity rows :", len(perplexity_df))
print("Coherence rows  :", len(coherence_df))

assert len(perplexity_df) == 495
assert len(coherence_df) == 495

assert (
    perplexity_df[
        ["Fold", "Topics"]
    ].duplicated().sum()
    == 0
)

assert (
    coherence_df[
        ["Fold", "Topics"]
    ].duplicated().sum()
    == 0
)

# ------------------------------------------------------------
# Raw coherence sign/range analysis
# ------------------------------------------------------------

coherence_columns = [
    "Coherence_UMass",
    "Coherence_CV",
    "Coherence_UCI",
    "Coherence_NPMI"
]

sign_rows = []

for column in coherence_columns:

    values = coherence_df[column]

    sign_rows.append({

        "Metric":
            column,

        "Minimum":
            values.min(),

        "Maximum":
            values.max(),

        "Mean":
            values.mean(),

        "Negative":
            int((values < 0).sum()),

        "Zero":
            int((values == 0).sum()),

        "Positive":
            int((values > 0).sum()),

        "NegativePct":
            100 * (values < 0).mean(),

        "PositivePct":
            100 * (values > 0).mean()
    })


coherence_sign_summary = pd.DataFrame(
    sign_rows
)

print("\n" + "=" * 80)
print("RAW COHERENCE SIGN SUMMARY")
print("=" * 80)

display(
    coherence_sign_summary
)


# ------------------------------------------------------------
# Merge raw metric results
# ------------------------------------------------------------

metric_master_df = pd.merge(

    perplexity_df,

    coherence_df,

    on=[
        "ExperimentID",
        "Fold",
        "Topics"
    ],

    how="inner",

    validate="one_to_one"
)


metric_master_df = (
    metric_master_df
    .sort_values(
        ["Fold", "Topics"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate merge
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("MASTER TABLE VALIDATION")
print("=" * 80)

print(
    "Rows              :",
    len(metric_master_df)
)

print(
    "Columns           :",
    len(metric_master_df.columns)
)

print(
    "Unique Fold-K     :",
    metric_master_df[
        ["Fold", "Topics"]
    ].drop_duplicates().shape[0]
)

print(
    "Missing values    :",
    metric_master_df.isna().sum().sum()
)


assert len(metric_master_df) == 495

assert (
    metric_master_df[
        ["Fold", "Topics"]
    ].drop_duplicates().shape[0]
    == 495
)

assert (
    metric_master_df
    .isna()
    .sum()
    .sum()
    == 0
)


# ------------------------------------------------------------
# Save raw master table
# ------------------------------------------------------------

RAW_MASTER_FILE = (
    METRIC_DIR
    / "ng04_raw_metric_master.csv"
)

metric_master_df.to_csv(
    RAW_MASTER_FILE,
    index=False
)


print("\nSaved:")
print(RAW_MASTER_FILE)

print("\n" + "=" * 80)
print("✓ RAW COHERENCE VALUES PRESERVED")
print("✓ PERPLEXITY + COHERENCE MERGED")
print("✓ 495 UNIQUE EXPERIMENTS VERIFIED")
print("=" * 80)

display(
    metric_master_df.head()
)

NG04-8R — RAW METRIC VALIDATION AND MERGE

Perplexity rows : 495
Coherence rows  : 495

RAW COHERENCE SIGN SUMMARY


,Metric,Minimum,Maximum,Mean,Negative,Zero,Positive,NegativePct,PositivePct
0,Coherence_UMass,-6.624327,-0.978500,-3.405733,495,0,0,100.000000,0.000000
1,Coherence_CV,0.482003,0.816420,0.631627,0,0,495,0.000000,100.000000
2,Coherence_UCI,-3.988233,1.343336,-1.459762,427,0,68,86.262626,13.737374
3,Coherence_NPMI,-0.077489,0.182041,0.020750,209,0,286,42.222222,57.777778



MASTER TABLE VALIDATION
Rows              : 495
Columns           : 11
Unique Fold-K     : 495
Missing values    : 0

Saved:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/metrics/ng04_raw_metric_master.csv

✓ RAW COHERENCE VALUES PRESERVED
✓ PERPLEXITY + COHERENCE MERGED
✓ 495 UNIQUE EXPERIMENTS VERIFIED


,ExperimentID,Fold,Topics,TrainLogPerplexity,TestLogPerplexity,TrainPerplexity,TestPerplexity,Coherence_UMass,Coherence_CV,Coherence_UCI,Coherence_NPMI
0,NG_F1_K2,1,2,-8.166942,-8.218584,287.405039,297.879338,-6.624327,0.482003,-3.988233,-0.077489
1,NG_F1_K3,1,3,-8.017795,-8.114469,259.177277,277.139657,-1.259477,0.679111,0.346130,0.106190
2,NG_F1_K4,1,4,-7.962156,-8.111846,249.372131,276.636202,-1.392020,0.674515,0.317378,0.092768
3,NG_F1_K5,1,5,-7.929313,-8.109783,243.759208,276.240885,-1.521834,0.713990,0.531896,0.112504
4,NG_F1_K6,1,6,-7.908885,-8.111695,240.331941,276.607296,-1.502607,0.711919,0.554199,0.104472


In [ ]:
# ============================================================
# NG04-8S — Exact Original Fold-wise NAC and NAP
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("NG04-8S — ORIGINAL NAC AND NAP TRANSFORMATIONS")
print("=" * 80)

# ------------------------------------------------------------
# Start from validated raw metric master table
# ------------------------------------------------------------

metric_df = pd.read_csv(
    RAW_MASTER_FILE
)

assert len(metric_df) == 495

assert (
    metric_df[
        ["Fold", "Topics"]
    ].duplicated().sum()
    == 0
)

# ============================================================
# 1. Absolute coherence values
# ============================================================

metric_df["AbsUMass"] = (
    metric_df["Coherence_UMass"].abs()
)

metric_df["AbsCV"] = (
    metric_df["Coherence_CV"].abs()
)

metric_df["AbsUCI"] = (
    metric_df["Coherence_UCI"].abs()
)

metric_df["AbsNPMI"] = (
    metric_df["Coherence_NPMI"].abs()
)

# ============================================================
# 2. Fold-wise NAC
# ============================================================

metric_df["NAC_UMass"] = (
    metric_df["AbsUMass"]
    /
    metric_df.groupby("Fold")[
        "AbsUMass"
    ].transform("max")
)

metric_df["NAC_CV"] = (
    metric_df["AbsCV"]
    /
    metric_df.groupby("Fold")[
        "AbsCV"
    ].transform("max")
)

metric_df["NAC_UCI"] = (
    metric_df["AbsUCI"]
    /
    metric_df.groupby("Fold")[
        "AbsUCI"
    ].transform("max")
)

metric_df["NAC_NPMI"] = (
    metric_df["AbsNPMI"]
    /
    metric_df.groupby("Fold")[
        "AbsNPMI"
    ].transform("max")
)

# ============================================================
# 3. Original NAP
# ============================================================
#
# Original notebook:
#
# MeanLogPerplexity
# -> absolute value
# -> divide by fold-wise maximum absolute log-perplexity
#
# For this replication, the corresponding held-out quantity is
# TestLogPerplexity computed in NG04-8O.
# ============================================================

metric_df["MeanLogPerplexity"] = (
    metric_df["TestLogPerplexity"]
)

metric_df["AbsLogPerplexity"] = (
    metric_df["MeanLogPerplexity"].abs()
)

metric_df["NAP"] = (
    metric_df["AbsLogPerplexity"]
    /
    metric_df.groupby("Fold")[
        "AbsLogPerplexity"
    ].transform("max")
)

# ============================================================
# Validation
# ============================================================

transformed_columns = [
    "NAC_UMass",
    "NAC_CV",
    "NAC_UCI",
    "NAC_NPMI",
    "NAP"
]

assert np.isfinite(
    metric_df[
        transformed_columns
    ].to_numpy()
).all()

for column in transformed_columns:

    assert (
        metric_df[column] >= 0
    ).all()

    assert (
        metric_df[column] <= 1
    ).all()


# Every fold should contain a maximum-normalized value of 1
for fold in range(1, 6):

    fold_df = metric_df[
        metric_df["Fold"] == fold
    ]

    for column in transformed_columns:

        assert np.isclose(
            fold_df[column].max(),
            1.0
        )


# ============================================================
# Save transformed master table
# ============================================================

TRANSFORMED_MASTER_FILE = (
    METRIC_DIR
    / "ng04_metric_master_with_nac_nap.csv"
)

metric_df.to_csv(
    TRANSFORMED_MASTER_FILE,
    index=False
)

print(
    "\nSaved:",
    TRANSFORMED_MASTER_FILE
)


# ============================================================
# Fold-wise maxima used for normalization
# ============================================================

normalization_summary = (
    metric_df
    .groupby("Fold")
    .agg(
        MaxAbsUMass=("AbsUMass", "max"),
        MaxAbsCV=("AbsCV", "max"),
        MaxAbsUCI=("AbsUCI", "max"),
        MaxAbsNPMI=("AbsNPMI", "max"),
        MaxAbsLogPerplexity=(
            "AbsLogPerplexity",
            "max"
        )
    )
)

print("\n" + "=" * 80)
print("FOLD-WISE NORMALIZATION DENOMINATORS")
print("=" * 80)

display(
    normalization_summary
)


# ============================================================
# Topic number selected by each transformed metric
# ============================================================

selection_rows = []

for fold in range(1, 6):

    fold_df = metric_df[
        metric_df["Fold"] == fold
    ]

    row = {
        "Fold": fold
    }

    for column in transformed_columns:

        selected_index = (
            fold_df[column]
            .idxmax()
        )

        row[column] = int(
            metric_df.loc[
                selected_index,
                "Topics"
            ]
        )

    selection_rows.append(
        row
    )


selection_summary = pd.DataFrame(
    selection_rows
)

print("\n" + "=" * 80)
print("K SELECTED BY ORIGINAL NAC/NAP")
print("=" * 80)

display(
    selection_summary
)


print("\n" + "=" * 80)
print("✓ ORIGINAL ABSOLUTE-VALUE TRANSFORMATION REPRODUCED")
print("✓ NAC NORMALIZED WITHIN EACH FOLD")
print("✓ NAP NORMALIZED WITHIN EACH FOLD")
print("✓ ALL TRANSFORMED VALUES ARE IN [0, 1]")
print("=" * 80)

display(
    metric_df[
        [
            "ExperimentID",
            "Fold",
            "Topics",

            "Coherence_UMass",
            "AbsUMass",
            "NAC_UMass",

            "Coherence_CV",
            "AbsCV",
            "NAC_CV",

            "Coherence_UCI",
            "AbsUCI",
            "NAC_UCI",

            "Coherence_NPMI",
            "AbsNPMI",
            "NAC_NPMI",

            "MeanLogPerplexity",
            "AbsLogPerplexity",
            "NAP"
        ]
    ].head(10)
)

NG04-8S — ORIGINAL NAC AND NAP TRANSFORMATIONS

Saved: /content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/metrics/ng04_metric_master_with_nac_nap.csv

FOLD-WISE NORMALIZATION DENOMINATORS


,MaxAbsUMass,MaxAbsCV,MaxAbsUCI,MaxAbsNPMI,MaxAbsLogPerplexity
Fold,,,,,
1,6.624327,0.713990,3.988233,0.112504,15.484621
2,4.826798,0.771700,3.101405,0.155436,11.452576
3,6.102812,0.763401,3.039293,0.131329,12.210208
4,4.529384,0.749012,2.940584,0.158527,11.562086
5,4.786634,0.816420,3.215946,0.182041,11.861954



K SELECTED BY ORIGINAL NAC/NAP


,Fold,NAC_UMass,NAC_CV,NAC_UCI,NAC_NPMI,NAP
0,1,2,5,2,5,100
1,2,84,4,84,4,100
2,3,3,10,91,7,100
3,4,94,7,98,2,100
4,5,80,5,86,5,100



✓ ORIGINAL ABSOLUTE-VALUE TRANSFORMATION REPRODUCED
✓ NAC NORMALIZED WITHIN EACH FOLD
✓ NAP NORMALIZED WITHIN EACH FOLD
✓ ALL TRANSFORMED VALUES ARE IN [0, 1]


,ExperimentID,Fold,Topics,Coherence_UMass,AbsUMass,NAC_UMass,Coherence_CV,AbsCV,NAC_CV,Coherence_UCI,AbsUCI,NAC_UCI,Coherence_NPMI,AbsNPMI,NAC_NPMI,MeanLogPerplexity,AbsLogPerplexity,NAP
0,NG_F1_K2,1,2,-6.624327,6.624327,1.000000,0.482003,0.482003,0.675084,-3.988233,3.988233,1.000000,-0.077489,0.077489,0.688772,-8.218584,8.218584,0.530758
1,NG_F1_K3,1,3,-1.259477,1.259477,0.190129,0.679111,0.679111,0.951148,0.346130,0.346130,0.086788,0.106190,0.106190,0.943884,-8.114469,8.114469,0.524034
2,NG_F1_K4,1,4,-1.392020,1.392020,0.210138,0.674515,0.674515,0.944711,0.317378,0.317378,0.079579,0.092768,0.092768,0.824573,-8.111846,8.111846,0.523865
3,NG_F1_K5,1,5,-1.521834,1.521834,0.229734,0.713990,0.713990,1.000000,0.531896,0.531896,0.133366,0.112504,0.112504,1.000000,-8.109783,8.109783,0.523731
4,NG_F1_K6,1,6,-1.502607,1.502607,0.226832,0.711919,0.711919,0.997099,0.554199,0.554199,0.138959,0.104472,0.104472,0.928611,-8.111695,8.111695,0.523855
5,NG_F1_K7,1,7,-1.848268,1.848268,0.279012,0.661607,0.661607,0.926633,0.029384,0.029384,0.007368,0.077049,0.077049,0.684861,-8.142549,8.142549,0.525847
6,NG_F1_K8,1,8,-2.852878,2.852878,0.430667,0.627860,0.627860,0.879367,-0.894749,0.894749,0.224347,0.038876,0.038876,0.345551,-8.171427,8.171427,0.527712
7,NG_F1_K9,1,9,-1.645164,1.645164,0.248352,0.691442,0.691442,0.968419,0.091416,0.091416,0.022921,0.089063,0.089063,0.791647,-8.178067,8.178067,0.528141
8,NG_F1_K10,1,10,-1.984278,1.984278,0.299544,0.685445,0.685445,0.960020,-0.150716,0.150716,0.037790,0.077739,0.077739,0.690991,-8.215569,8.215569,0.530563
9,NG_F1_K11,1,11,-1.840151,1.840151,0.277787,0.702560,0.702560,0.983991,0.168077,0.168077,0.042143,0.090852,0.090852,0.807549,-8.323661,8.323661,0.537544


In [ ]:
# ============================================================
# NG04-8T — Raw vs Transformed Ranking Diagnostics
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 80)
print("NG04-8T — RAW VS TRANSFORMED RANKING DIAGNOSTICS")
print("=" * 80)

metric_df = pd.read_csv(
    TRANSFORMED_MASTER_FILE
)

# ------------------------------------------------------------
# Metric definitions
# ------------------------------------------------------------
#
# For coherence:
#   larger raw coherence = better
#
# For log-perplexity:
#   larger log-perplexity = better
#   because -8 is better than -12.
#
# Transformed NAC/NAP are selected by maximum.
# ------------------------------------------------------------

metric_pairs = {
    "NAC_UMass": (
        "Coherence_UMass",
        "NAC_UMass"
    ),

    "NAC_CV": (
        "Coherence_CV",
        "NAC_CV"
    ),

    "NAC_UCI": (
        "Coherence_UCI",
        "NAC_UCI"
    ),

    "NAC_NPMI": (
        "Coherence_NPMI",
        "NAC_NPMI"
    ),

    "NAP": (
        "MeanLogPerplexity",
        "NAP"
    )
}

diagnostic_rows = []

for fold in range(1, 6):

    fold_df = (
        metric_df[
            metric_df["Fold"] == fold
        ]
        .copy()
        .sort_values("Topics")
        .reset_index(drop=True)
    )

    for metric_name, (
        raw_column,
        transformed_column
    ) in metric_pairs.items():

        # ----------------------------------------------------
        # Original raw ranking
        # rank 1 = best
        # ----------------------------------------------------

        raw_rank = (
            fold_df[raw_column]
            .rank(
                ascending=False,
                method="average"
            )
        )

        # ----------------------------------------------------
        # Transformed ranking
        # rank 1 = best
        # ----------------------------------------------------

        transformed_rank = (
            fold_df[transformed_column]
            .rank(
                ascending=False,
                method="average"
            )
        )

        # ----------------------------------------------------
        # Best K under raw metric
        # ----------------------------------------------------

        raw_best_index = (
            fold_df[raw_column]
            .idxmax()
        )

        raw_best_k = int(
            fold_df.loc[
                raw_best_index,
                "Topics"
            ]
        )

        # ----------------------------------------------------
        # Best K under transformed metric
        # ----------------------------------------------------

        transformed_best_index = (
            fold_df[transformed_column]
            .idxmax()
        )

        transformed_best_k = int(
            fold_df.loc[
                transformed_best_index,
                "Topics"
            ]
        )

        # ----------------------------------------------------
        # Ranking agreement
        # ----------------------------------------------------

        spearman_rho, _ = spearmanr(
            raw_rank,
            transformed_rank
        )

        mean_abs_rank_difference = float(
            np.mean(
                np.abs(
                    raw_rank
                    - transformed_rank
                )
            )
        )

        max_abs_rank_difference = float(
            np.max(
                np.abs(
                    raw_rank
                    - transformed_rank
                )
            )
        )

        different_rank_positions = int(
            np.sum(
                raw_rank
                != transformed_rank
            )
        )

        diagnostic_rows.append({

            "Fold":
                fold,

            "Metric":
                metric_name,

            "RawMetric":
                raw_column,

            "TransformedMetric":
                transformed_column,

            "RawBestK":
                raw_best_k,

            "TransformedBestK":
                transformed_best_k,

            "BestKAgreement":
                int(
                    raw_best_k
                    == transformed_best_k
                ),

            "SpearmanRho":
                spearman_rho,

            "DifferentRankPositions":
                different_rank_positions,

            "MeanAbsoluteRankDifference":
                mean_abs_rank_difference,

            "MaxAbsoluteRankDifference":
                max_abs_rank_difference
        })


ranking_diagnostic_df = pd.DataFrame(
    diagnostic_rows
)

# ============================================================
# Save fold-level diagnostics
# ============================================================

RANKING_DIAGNOSTIC_FILE = (
    METRIC_DIR
    / "ng04_raw_vs_transformed_ranking.csv"
)

ranking_diagnostic_df.to_csv(
    RANKING_DIAGNOSTIC_FILE,
    index=False
)


# ============================================================
# Display fold-level results
# ============================================================

print("\n" + "=" * 80)
print("FOLD-LEVEL RANKING DIAGNOSTICS")
print("=" * 80)

display(
    ranking_diagnostic_df
)


# ============================================================
# Metric-level summary
# ============================================================

ranking_summary_df = (
    ranking_diagnostic_df
    .groupby("Metric")
    .agg(
        Folds=(
            "Fold",
            "count"
        ),

        BestKAgreements=(
            "BestKAgreement",
            "sum"
        ),

        BestKAgreementPct=(
            "BestKAgreement",
            lambda x: 100 * x.mean()
        ),

        MeanSpearmanRho=(
            "SpearmanRho",
            "mean"
        ),

        MeanDifferentRankPositions=(
            "DifferentRankPositions",
            "mean"
        ),

        MeanAbsoluteRankDifference=(
            "MeanAbsoluteRankDifference",
            "mean"
        ),

        MaximumRankDifference=(
            "MaxAbsoluteRankDifference",
            "max"
        )
    )
    .reset_index()
)


RANKING_SUMMARY_FILE = (
    METRIC_DIR
    / "ng04_raw_vs_transformed_ranking_summary.csv"
)

ranking_summary_df.to_csv(
    RANKING_SUMMARY_FILE,
    index=False
)

print("\n" + "=" * 80)
print("METRIC-LEVEL SUMMARY")
print("=" * 80)

display(
    ranking_summary_df
)


print("\nSaved:")
print(RANKING_DIAGNOSTIC_FILE)
print(RANKING_SUMMARY_FILE)

print("\n" + "=" * 80)
print("✓ RAW AND TRANSFORMED RANKINGS COMPARED")
print("✓ FOLD-LEVEL BEST-K AGREEMENT MEASURED")
print("✓ SPEARMAN RANK AGREEMENT MEASURED")
print("✓ RANK-POSITION DISTORTION MEASURED")
print("=" * 80)

NG04-8T — RAW VS TRANSFORMED RANKING DIAGNOSTICS

FOLD-LEVEL RANKING DIAGNOSTICS


,Fold,Metric,RawMetric,TransformedMetric,RawBestK,TransformedBestK,BestKAgreement,SpearmanRho,DifferentRankPositions,MeanAbsoluteRankDifference,MaxAbsoluteRankDifference
0,1,NAC_UMass,Coherence_UMass,NAC_UMass,3,2,0,-1.000000,98,49.494949,98.0
1,1,NAC_CV,Coherence_CV,NAC_CV,5,5,1,1.000000,0,0.000000,0.0
2,1,NAC_UCI,Coherence_UCI,NAC_UCI,6,2,0,-0.984552,98,49.494949,98.0
3,1,NAC_NPMI,Coherence_NPMI,NAC_NPMI,5,5,1,0.348918,82,24.484848,81.0
4,1,NAP,MeanLogPerplexity,NAP,5,100,0,-1.000000,98,49.494949,98.0
5,2,NAC_UMass,Coherence_UMass,NAC_UMass,2,84,0,-1.000000,98,49.494949,98.0
6,2,NAC_CV,Coherence_CV,NAC_CV,4,4,1,1.000000,0,0.000000,0.0
7,2,NAC_UCI,Coherence_UCI,NAC_UCI,4,84,0,-0.982362,98,49.494949,98.0
8,2,NAC_NPMI,Coherence_NPMI,NAC_NPMI,4,4,1,0.514051,74,20.000000,74.0
9,2,NAP,MeanLogPerplexity,NAP,5,100,0,-1.000000,98,49.494949,98.0



METRIC-LEVEL SUMMARY


,Metric,Folds,BestKAgreements,BestKAgreementPct,MeanSpearmanRho,MeanDifferentRankPositions,MeanAbsoluteRankDifference,MaximumRankDifference
0,NAC_CV,5,5,100.0,1.000000,0.0,0.000000,0.0
1,NAC_NPMI,5,5,100.0,0.535649,71.4,19.466667,81.0
2,NAC_UCI,5,0,0.0,-0.969237,98.0,49.494949,98.0
3,NAC_UMass,5,0,0.0,-1.000000,98.0,49.494949,98.0
4,NAP,5,0,0.0,-1.000000,98.0,49.494949,98.0



Saved:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/metrics/ng04_raw_vs_transformed_ranking.csv
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/metrics/ng04_raw_vs_transformed_ranking_summary.csv

✓ RAW AND TRANSFORMED RANKINGS COMPARED
✓ FOLD-LEVEL BEST-K AGREEMENT MEASURED
✓ SPEARMAN RANK AGREEMENT MEASURED
✓ RANK-POSITION DISTORTION MEASURED


In [ ]:
# ============================================================
# NG04-8U — Correct NAP Ranking Direction
#          NAP is a MINIMIZATION criterion
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 80)
print("NG04-8U — CORRECT NAP RANKING DIAGNOSTIC")
print("=" * 80)

METRIC_FILE = (
    BASE_DIR
    / "metrics"
    / "ng04_metric_master_with_nac_nap.csv"
)

df = pd.read_csv(METRIC_FILE)

assert len(df) == 495

# ============================================================
# Correct direction:
#
# MeanLogPerplexity:
# higher / less negative = better
#
# NAP:
# smaller = better
#
# Therefore:
# raw ranking      -> descending MeanLogPerplexity
# transformed rank -> ascending NAP
# ============================================================

rows = []

for fold in sorted(df["Fold"].unique()):

    fold_df = (
        df[df["Fold"] == fold]
        .copy()
        .sort_values("Topics")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Raw log-perplexity ranking:
    # larger / less negative is better
    # --------------------------------------------------------

    fold_df["RawRank"] = (
        fold_df["MeanLogPerplexity"]
        .rank(
            ascending=False,
            method="min"
        )
    )

    # --------------------------------------------------------
    # NAP ranking:
    # smaller is better
    # --------------------------------------------------------

    fold_df["NAPRank"] = (
        fold_df["NAP"]
        .rank(
            ascending=True,
            method="min"
        )
    )

    rho, _ = spearmanr(
        fold_df["RawRank"],
        fold_df["NAPRank"]
    )

    different_positions = (
        fold_df["RawRank"]
        !=
        fold_df["NAPRank"]
    ).sum()

    mean_abs_rank_difference = (
        fold_df["RawRank"]
        .sub(fold_df["NAPRank"])
        .abs()
        .mean()
    )

    max_abs_rank_difference = (
        fold_df["RawRank"]
        .sub(fold_df["NAPRank"])
        .abs()
        .max()
    )

    raw_best_k = int(
        fold_df.loc[
            fold_df[
                "MeanLogPerplexity"
            ].idxmax(),
            "Topics"
        ]
    )

    nap_best_k = int(
        fold_df.loc[
            fold_df[
                "NAP"
            ].idxmin(),
            "Topics"
        ]
    )

    rows.append(
        {
            "Fold": fold,
            "SpearmanRho":
                rho,

            "DifferentRankPositions":
                int(different_positions),

            "MeanAbsRankDifference":
                mean_abs_rank_difference,

            "MaxAbsRankDifference":
                max_abs_rank_difference,

            "RawBestK":
                raw_best_k,

            "NAPBestK":
                nap_best_k,

            "BestKAgreement":
                raw_best_k == nap_best_k
        }
    )

nap_corrected_diagnostic = pd.DataFrame(
    rows
)

# ============================================================
# Display
# ============================================================

print("\nCorrect NAP ranking diagnostic:")
display(
    nap_corrected_diagnostic
)

print(
    "\nBest-K agreement:",
    int(
        nap_corrected_diagnostic[
            "BestKAgreement"
        ].sum()
    ),
    "/",
    len(nap_corrected_diagnostic)
)

print(
    "Mean Spearman rho:",
    nap_corrected_diagnostic[
        "SpearmanRho"
    ].mean()
)

print(
    "Mean absolute rank difference:",
    nap_corrected_diagnostic[
        "MeanAbsRankDifference"
    ].mean()
)

# ============================================================
# Save
# ============================================================

OUTPUT_FILE = (
    BASE_DIR
    / "metrics"
    / "ng04_nap_corrected_ranking_diagnostic.csv"
)

nap_corrected_diagnostic.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\nSaved:")
print(OUTPUT_FILE)

print("\n" + "=" * 80)
print("✓ NAP TREATED AS MINIMIZATION CRITERION")
print("✓ RAW AND TRANSFORMED RANKINGS COMPARED IN CORRECT DIRECTIONS")
print("=" * 80)

NG04-8U — CORRECT NAP RANKING DIAGNOSTIC

Correct NAP ranking diagnostic:


,Fold,SpearmanRho,DifferentRankPositions,MeanAbsRankDifference,MaxAbsRankDifference,RawBestK,NAPBestK,BestKAgreement
0,1,1.0,0,0.0,0.0,5,5,True
1,2,1.0,0,0.0,0.0,5,5,True
2,3,1.0,0,0.0,0.0,5,5,True
3,4,1.0,0,0.0,0.0,4,4,True
4,5,1.0,0,0.0,0.0,7,7,True



Best-K agreement: 5 / 5
Mean Spearman rho: 0.9999999999999998
Mean absolute rank difference: 0.0

Saved:
/content/drive/MyDrive/LDAChatGPT/NewsGroupsTopicEval/metrics/ng04_nap_corrected_ranking_diagnostic.csv

✓ NAP TREATED AS MINIMIZATION CRITERION
✓ RAW AND TRANSFORMED RANKINGS COMPARED IN CORRECT DIRECTIONS
